# 📘 Worksheet Generator — Complete Notebook

A complete **Worksheet Generator** that runs in Google Colab. Teachers can generate
professional, grade-appropriate worksheets (Grade 1–10) with a school's own branding,
in three different ways:

- **Method 1** — Upload a file and pick a specific chapter/topic/page, OR select
  Grade + Subject + Topic directly (no file needed)
- **Method 2** — Upload any file and let AI automatically generate the worksheet
- **Method 3** — Paste text (e.g. copied from ChatGPT) and let AI clean it up and
  build the worksheet

**Shared core engine features (used by all three methods):**
- School setup & branding (name, logo with adjustable size, theme colors, cover page style)
- Grade-aware themes: Grade 1–5 = colorful/cartoonic (with fun icons), Grade 6–10 = professional/minimal
- Question types: Multiple Choice, Fill in the Blanks, Match the Column, Short Questions,
  Detailed Questions — any combination, any quantity
- Layout controls: questions per page (unlimited), spacing, number of copies (unlimited),
  randomization (same order or shuffled per copy), binding gap
- Student info fields on every page: Name, Class, Roll No, Date
- **Accurate Answer Key** for every question type — teacher chooses: same file (last page),
  a separate teacher-only file, or both
- Full Urdu language support — pure Urdu script only, never Roman Urdu
- Preview before generating
- Export to PDF or Word, teacher's choice
- Delete/skip any option that isn't needed

➡️ **Run each cell from top to bottom, in order (Shift+Enter).**


## 🔧 Core Engine

### Step 1 — Install Packages (run once)

In [ ]:
# Run this cell only once (to install packages)
!pip install weasyprint python-docx ipywidgets arabic-reshaper python-bidi Pillow -q
print("✅ All packages installed successfully!")


### Step 2 — Import Libraries

In [ ]:
import base64
import io
import random
import re
import uuid
from datetime import date

import ipywidgets as widgets
from IPython.display import display, HTML, FileLink, clear_output
from PIL import Image
import arabic_reshaper
from bidi.algorithm import get_display

from weasyprint import HTML as WeasyHTML
from docx import Document
from docx.shared import Pt, Inches, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT

print("✅ Libraries imported. Core engine ready to configure.")


### Step 3 — Global Config
This holds the system's shared settings and bilingual labels. All methods below reuse it — please don't remove this cell.

In [ ]:
# ============================================================
# GLOBAL CONFIG — this dict holds all settings for the whole system
# (Later methods reuse this same CONFIG, so don't modify this cell)
# ============================================================

CONFIG = {
    "school_name": "",
    "school_logo_base64": None,     # base64 string of uploaded logo
    "logo_size_px": 70,              # teacher adjustable
    "theme_mode": "auto",            # "auto" | "colorful" | "professional"
    "primary_color": "#2E86AB",
    "accent_color": "#F6C90E",
    "cover_page_style": "modern",    # "modern" | "classic" | "minimal"
}


def fix_urdu(text: str) -> str:
    """
    Har jagah jahan Urdu text worksheet par likhna ho, isay is function
    se guzaarein — ye reshaping + RTL direction sahi karta hai taake
    letters tootay na hon aur order sahi (right-to-left) rahe.
    """
    if text is None:
        return ""
    # Agar text mein Urdu/Arabic characters hain to reshape karo
    if re.search(r'[\u0600-\u06FF]', text):
        reshaped = arabic_reshaper.reshape(text)
        return get_display(reshaped)
    return text


def is_urdu_text(text: str) -> bool:
    return bool(re.search(r'[\u0600-\u06FF]', text or ""))



# ------------------------------------------------------------
# LABELS — bilingual (English / pure Urdu). When the worksheet's
# language is Urdu, no label will ever appear in Roman Urdu —
# only pure Urdu script is used.
# ------------------------------------------------------------
LABELS = {
    "en": {
        "name": "Name", "class": "Class", "roll_no": "Roll No", "date": "Date",
        "answer_key": "Answer Key", "grade": "Grade", "subject": "Subject",
        "question": "Question", "correct_answer": "Correct Answer",
        "model_answer": "Model Answer", "teacher_copy": "Teacher Copy — Answer Key",
    },
    "ur": {
        "name": "نام", "class": "جماعت", "roll_no": "رول نمبر", "date": "تاریخ",
        "answer_key": "جوابات", "grade": "جماعت", "subject": "مضمون",
        "question": "سوال", "correct_answer": "درست جواب",
        "model_answer": "نمونہ جواب", "teacher_copy": "اساتذہ کے لیے — جوابات",
    },
}


def L(key: str, lang: str = "en") -> str:
    """Label lookup. Urdu labels always come in pure Urdu script (never Roman Urdu)."""
    label = LABELS.get(lang, LABELS["en"]).get(key, key)
    return fix_urdu(label) if lang == "ur" else label


def image_to_base64(file_bytes: bytes) -> str:
    img = Image.open(io.BytesIO(file_bytes))
    buf = io.BytesIO()
    fmt = img.format if img.format else "PNG"
    img.save(buf, format=fmt)
    encoded = base64.b64encode(buf.getvalue()).decode("utf-8")
    mime = f"image/{fmt.lower()}"
    return f"data:{mime};base64,{encoded}"


print("✅ CONFIG initialized:", CONFIG)


### Step 4 — 🏫 School Setup & Branding
Enter the school name, upload a logo, and choose theme colors. Remember to click **Save**.

In [ ]:
# ============================================================
# SCHOOL SETUP & BRANDING — Teacher/Admin enters the school's
# details here. This is a one-time setup.
# ============================================================

school_name_box = widgets.Text(
    value='', placeholder='Enter school name (e.g. Roots Millennium School)',
    description='School Name:', layout=widgets.Layout(width='450px'),
    style={'description_width': '120px'}
)

logo_uploader = widgets.FileUpload(
    accept='image/*', multiple=False, description='Upload Logo'
)

logo_size_slider = widgets.IntSlider(
    value=70, min=30, max=200, step=5,
    description='Logo Size (px):', style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

theme_mode_dd = widgets.Dropdown(
    options=[('Auto (grade ke hisaab se)', 'auto'),
             ('Hamesha Colorful/Cartoonic', 'colorful'),
             ('Hamesha Professional/Minimal', 'professional')],
    value='auto', description='Theme Mode:', style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

primary_color_picker = widgets.ColorPicker(
    concise=False, description='Primary Color:', value='#2E86AB',
    style={'description_width': '120px'}
)

accent_color_picker = widgets.ColorPicker(
    concise=False, description='Accent Color:', value='#F6C90E',
    style={'description_width': '120px'}
)

cover_style_dd = widgets.Dropdown(
    options=[('Modern', 'modern'), ('Classic', 'classic'), ('Minimal', 'minimal')],
    value='modern', description='Cover Style:', style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

save_setup_btn = widgets.Button(description='💾 Save School Setup', button_style='success')
setup_output = widgets.Output()


def save_school_setup(b):
    with setup_output:
        clear_output()
        CONFIG["school_name"] = school_name_box.value.strip()
        CONFIG["logo_size_px"] = logo_size_slider.value
        CONFIG["theme_mode"] = theme_mode_dd.value
        CONFIG["primary_color"] = primary_color_picker.value
        CONFIG["accent_color"] = accent_color_picker.value
        CONFIG["cover_page_style"] = cover_style_dd.value

        if logo_uploader.value:
            uploaded_file = list(logo_uploader.value.values())[0] if isinstance(logo_uploader.value, dict) else logo_uploader.value[0]
            file_bytes = uploaded_file['content'] if isinstance(uploaded_file, dict) else uploaded_file.content
            CONFIG["school_logo_base64"] = image_to_base64(bytes(file_bytes))
            print("✅ Logo uploaded successfully.")

        if not CONFIG["school_name"]:
            print("⚠️  Please enter the school name — it's required.")
            return

        print("✅ School setup saved!")
        print(f"   School Name : {CONFIG['school_name']}")
        print(f"   Theme Mode  : {CONFIG['theme_mode']}")
        print(f"   Colors      : {CONFIG['primary_color']} / {CONFIG['accent_color']}")
        print(f"   Logo        : {'Uploaded ✅' if CONFIG['school_logo_base64'] else 'Not uploaded'}")


save_setup_btn.on_click(save_school_setup)

display(widgets.HTML("<h3>🏫 School Setup & Branding</h3>"))
display(widgets.VBox([
    school_name_box,
    widgets.HBox([logo_uploader, widgets.Label("(PNG/JPG, transparent background behtar rahega)")]),
    logo_size_slider,
    theme_mode_dd,
    widgets.HBox([primary_color_picker, accent_color_picker]),
    cover_style_dd,
    save_setup_btn,
    setup_output
]))


### Step 5 — Theme Engine (colorful vs. professional, based on grade)

In [ ]:
# ============================================================
# THEME ENGINE — decides the look based on grade
# Grade 1-5  -> Colorful / Cartoonic (designed to attract young children)
# Grade 6-10 -> Professional / Clean / Minimal
# ============================================================

def get_theme_for_grade(grade: int, forced_mode: str = "auto") -> str:
    if forced_mode in ("colorful", "professional"):
        return forced_mode
    return "colorful" if int(grade) <= 5 else "professional"


def base_css(primary, accent):
    return f"""
    @font-face {{
        font-family: 'NotoUrdu';
        src: url('https://fonts.gstatic.com/s/notonastaliqurdu/v14/LhWnMVvVU-Fc2LzX6DUb5jgs2vzD-oSaqXhTsB4T.ttf');
    }}
    * {{ box-sizing: border-box; }}
    body {{
        font-family: 'Segoe UI', Arial, sans-serif;
        margin: 0;
        color: #2b2b2b;
    }}
    .urdu-text {{
        font-family: 'NotoUrdu', 'Segoe UI', serif;
        direction: rtl;
        text-align: right;
        font-size: 16px;
        line-height: 2;
    }}
    .page {{
        page-break-after: always;
        padding: 20px 28px;
        position: relative;
        min-height: 100%;
    }}
    .page:last-child {{ page-break-after: auto; }}

    .header-bar {{
        display: flex;
        align-items: center;
        justify-content: space-between;
        border-bottom: 3px solid {primary};
        padding-bottom: 8px;
        margin-bottom: 14px;
    }}
    .header-left {{ display: flex; align-items: center; gap: 10px; }}
    .school-name {{ font-size: 18px; font-weight: 700; color: {primary}; }}
    .logo-img {{ object-fit: contain; }}

    .student-info-bar {{
        display: flex;
        flex-wrap: wrap;
        gap: 18px;
        background: #f4f7fb;
        border: 1px dashed {primary};
        border-radius: 8px;
        padding: 8px 14px;
        margin-bottom: 16px;
        font-size: 13px;
    }}
    .student-info-bar span b {{ color: {primary}; }}

    .question-block {{
        margin-bottom: 18px;
        padding-bottom: 10px;
    }}
    .q-number {{
        font-weight: 700;
        color: {primary};
        margin-right: 6px;
    }}
    .options-row {{
        display: flex;
        flex-wrap: wrap;
        gap: 14px;
        margin-top: 6px;
        margin-left: 22px;
    }}
    .option-item {{
        border: 1px solid #ccc;
        border-radius: 6px;
        padding: 4px 10px;
        font-size: 13px;
    }}
    .fill-blank-line {{
        display: inline-block;
        border-bottom: 1.5px solid #333;
        min-width: 120px;
        margin: 0 4px;
    }}
    .match-column-table {{
        width: 90%;
        margin: 8px auto 0 22px;
        border-collapse: collapse;
    }}
    .match-column-table td {{
        border: 1px solid #bbb;
        padding: 6px 10px;
        font-size: 13px;
    }}
    .answer-lines .line {{
        border-bottom: 1px solid #999;
        height: 22px;
        margin: 0 22px 4px 22px;
    }}
    .footer-bar {{
        position: absolute;
        bottom: 10px;
        left: 28px;
        right: 28px;
        display: flex;
        justify-content: space-between;
        font-size: 11px;
        color: #777;
        border-top: 1px solid #ddd;
        padding-top: 4px;
    }}
    .binding-gap {{
        width: {{BINDING_GAP}}px;
    }}
    .answer-key-page {{
        background: #fbfbfb;
    }}
    .answer-key-list .answer-key-item {{
        padding: 6px 0;
        border-bottom: 1px dashed #ccc;
        font-size: 14px;
    }}
    """


def theme_extra_css(theme: str, accent):
    if theme == "colorful":
        return f"""
        body {{ background: #fffdf6; }}
        .header-bar {{ background: linear-gradient(90deg, {accent}33, transparent); border-radius: 10px; }}
        .question-block {{
            background: #ffffff;
            border: 2px solid {accent};
            border-radius: 14px;
            padding: 12px 16px;
        }}
        .q-number {{ font-size: 16px; }}
        .option-item {{ background: #fff8e1; border-color: {accent}; border-radius: 14px; }}
        """
    else:  # professional
        return """
        body { background: #ffffff; }
        .question-block { border-bottom: 1px solid #e2e2e2; }
        .option-item { background: #fafafa; border-radius: 4px; }
        """


# Simple emoji-based "real image" icons for younger grades (1-3)
# (work offline, no internet required)
GRADE_ICON_MAP = {
    "apple": "🍎", "banana": "🍌", "cat": "🐱", "dog": "🐶", "monkey": "🐵",
    "elephant": "🐘", "lion": "🦁", "fish": "🐟", "bird": "🐦", "sun": "☀️",
    "moon": "🌙", "star": "⭐", "flower": "🌸", "tree": "🌳", "car": "🚗",
    "ball": "⚽", "book": "📕", "house": "🏠", "cow": "🐄", "chicken": "🐔",
}


def add_icon_if_available(word: str) -> str:
    return GRADE_ICON_MAP.get(word.strip().lower(), "")


print("✅ Theme engine loaded (colorful + professional).")


### Step 6 — Cover Page + Header/Footer Builders

In [ ]:
# ============================================================
# COVER PAGE + HEADER/FOOTER BUILDERS
# ============================================================

def build_logo_html(size_px=None):
    if not CONFIG.get("school_logo_base64"):
        return ""
    size = size_px or CONFIG.get("logo_size_px", 70)
    return f'<img class="logo-img" src="{CONFIG["school_logo_base64"]}" style="width:{size}px;height:{size}px;">'


def build_cover_page_html(worksheet_title, grade, subject, style=None, lang="en"):
    style = style or CONFIG.get("cover_page_style", "modern")
    primary = CONFIG.get("primary_color", "#2E86AB")
    accent = CONFIG.get("accent_color", "#F6C90E")
    logo = build_logo_html(size_px=130)
    school_name = CONFIG.get("school_name", "")
    title_disp = fix_urdu(worksheet_title) if lang == "ur" else worksheet_title
    grade_lbl, subj_lbl = L("grade", lang), L("subject", lang)
    title_css = "urdu-text" if lang == "ur" else ""

    if style == "modern":
        body = f"""
        <div style="text-align:center; padding-top:80px;">
            {logo}
            <h1 style="color:{primary}; font-size:34px; margin-top:20px;">{school_name}</h1>
            <div style="width:120px; height:4px; background:{accent}; margin:14px auto;"></div>
            <h2 class="{title_css}" style="color:#333; font-size:26px;">{title_disp}</h2>
            <p style="font-size:18px; color:#555;">{grade_lbl}: {grade} &nbsp;|&nbsp; {subj_lbl}: {subject}</p>
        </div>
        """
    elif style == "classic":
        body = f"""
        <div style="text-align:center; border:6px double {primary}; margin:40px; padding:60px 30px;">
            {logo}
            <h1 style="font-family:Georgia, serif; color:{primary};">{school_name}</h1>
            <h2 class="{title_css}" style="font-family:Georgia, serif;">{title_disp}</h2>
            <p>{grade_lbl}: {grade} | {subj_lbl}: {subject}</p>
        </div>
        """
    else:  # minimal
        body = f"""
        <div style="padding-top:100px; padding-left:60px;">
            {logo}
            <h1 style="color:{primary}; font-weight:300;">{school_name}</h1>
            <h3 class="{title_css}" style="color:#666; font-weight:300;">{title_disp} — {grade_lbl} {grade}, {subject}</h3>
        </div>
        """
    return f'<div class="page" style="page-break-after: always;">{body}</div>'


def build_student_info_bar(lang="en"):
    css_class = "urdu-text" if lang == "ur" else ""
    return f"""
    <div class="student-info-bar {css_class}">
        <span><b>{L('name', lang)}:</b> _____________________</span>
        <span><b>{L('class', lang)}:</b> _______</span>
        <span><b>{L('roll_no', lang)}:</b> _______</span>
        <span><b>{L('date', lang)}:</b> _______</span>
    </div>
    """


def build_page_header(worksheet_title, lang="en"):
    title_class = "urdu-text" if lang == "ur" else ""
    return f"""
    <div class="header-bar">
        <div class="header-left">
            {build_logo_html()}
            <div class="school-name">{CONFIG.get("school_name","")}</div>
        </div>
        <div class="{title_class}" style="font-size:14px; font-weight:600;">{fix_urdu(worksheet_title) if lang=='ur' else worksheet_title}</div>
    </div>
    {build_student_info_bar(lang)}
    """


def build_page_footer(page_num, total_pages, lang="en"):
    page_word = "صفحہ" if lang == "ur" else "Page"
    of_word = "از" if lang == "ur" else "of"
    page_word = fix_urdu(page_word) if lang == "ur" else page_word
    return f"""
    <div class="footer-bar">
        <span>{CONFIG.get("school_name","")}</span>
        <span>{page_word} {page_num} {of_word} {total_pages}</span>
    </div>
    """


print("✅ Cover page & header/footer builders ready.")


### Step 7 — Question Type Renderers (MCQ, Fill Blank, Match Column, Short, Detailed)

In [ ]:
# ============================================================
# QUESTION RENDERERS
# Every question dict should follow this shape:
# {
#   "type": "mcq" | "fill_blank" | "match_column" | "short" | "detailed",
#   "text": "the question's text",
#   "options": [...]        # for mcq
#   "blanks": 1              # for fill_blank (how many blanks)
#   "left": [...], "right": [...]   # for match_column
#   "lines": 3               # for short/detailed (answer space in lines)
#   "grade": 3               # used for adding an icon (optional)
# }
# ============================================================

def _q_text(q, q_num, theme):
    text = q.get("text", "")
    urdu = is_urdu_text(text)
    css_class = "urdu-text" if urdu else ""
    display_text = fix_urdu(text) if urdu else text
    icon = ""
    if theme == "colorful":
        icon = add_icon_if_available(text.split()[-1]) if text else ""
        icon = f" {icon}" if icon else ""
    return f'<div><span class="q-number">{q_num}.</span><span class="{css_class}">{display_text}{icon}</span></div>'


def render_mcq(q, q_num, theme):
    header = _q_text(q, q_num, theme)
    options_html = "".join(
        f'<div class="option-item">{chr(97+i)}) {opt}</div>'
        for i, opt in enumerate(q.get("options", []))
    )
    return f'<div class="question-block">{header}<div class="options-row">{options_html}</div></div>'


def render_fill_blank(q, q_num, theme):
    header = _q_text(q, q_num, theme)
    blanks = q.get("blanks", 1)
    blank_html = " ".join('<span class="fill-blank-line">&nbsp;</span>' for _ in range(blanks))
    return f'<div class="question-block">{header}<div style="margin-top:6px; margin-left:22px;">{blank_html}</div></div>'


def render_match_column(q, q_num, theme):
    header = _q_text(q, q_num, theme)
    left = q.get("left", [])
    right = q.get("right", [])
    # Shuffle ONCE and store on the question dict, so the printed worksheet
    # so the printed worksheet and its answer key always match (stay accurate)
    if "_shuffled_right" not in q:
        right_shuffled = right[:]
        random.shuffle(right_shuffled)
        q["_shuffled_right"] = right_shuffled
    right_shuffled = q["_shuffled_right"]
    rows = "".join(
        f"<tr><td>{l}</td><td style='width:40px;text-align:center;'>—</td><td>{r}</td></tr>"
        for l, r in zip(left, right_shuffled)
    )
    table = f'<table class="match-column-table">{rows}</table>'
    return f'<div class="question-block">{header}{table}</div>'


def render_short_or_detailed(q, q_num, theme):
    header = _q_text(q, q_num, theme)
    lines = q.get("lines", 3 if q.get("type") == "short" else 6)
    lines_html = "".join('<div class="line"></div>' for _ in range(lines))
    return f'<div class="question-block">{header}<div class="answer-lines" style="margin-top:8px;">{lines_html}</div></div>'


RENDERERS = {
    "mcq": render_mcq,
    "fill_blank": render_fill_blank,
    "match_column": render_match_column,
    "short": render_short_or_detailed,
    "detailed": render_short_or_detailed,
}


def render_question(q, q_num, theme):
    renderer = RENDERERS.get(q.get("type"))
    if not renderer:
        return f'<div class="question-block">Unknown question type: {q.get("type")}</div>'
    return renderer(q, q_num, theme)


# ============================================================
# ANSWER KEY RENDERERS
# Every question type should have an "answer" field:
#   mcq          -> "answer": "Apple"           (the correct option's text)
#   fill_blank   -> "answer": "cat"  (ya list agar multiple blanks: ["cat","dog"])
#   match_column -> answer is derived automatically from the original left/right order
#   short        -> "answer": "Model/expected answer text"
#   detailed     -> "answer": "Model/expected answer text"
# ============================================================

def _answer_line(q_num, content, lang="en"):
    css = "urdu-text" if lang == "ur" else ""
    return f'<div class="answer-key-item"><span class="q-number">{q_num}.</span> <span class="{css}">{content}</span></div>'


def render_answer_mcq(q, q_num, lang):
    ans = q.get("answer", "—")
    ans_disp = fix_urdu(ans) if lang == "ur" else ans
    return _answer_line(q_num, ans_disp, lang)


def render_answer_fill_blank(q, q_num, lang):
    ans = q.get("answer", "—")
    if isinstance(ans, list):
        ans = ", ".join(ans)
    ans_disp = fix_urdu(ans) if lang == "ur" else ans
    return _answer_line(q_num, ans_disp, lang)


def render_answer_match_column(q, q_num, lang):
    left = q.get("left", [])
    right = q.get("right", [])  # original (correct) order
    pairs = ", ".join(f"{l} → {r}" for l, r in zip(left, right))
    pairs_disp = fix_urdu(pairs) if lang == "ur" else pairs
    return _answer_line(q_num, pairs_disp, lang)


def render_answer_short_detailed(q, q_num, lang):
    ans = q.get("answer", "—")
    ans_disp = fix_urdu(ans) if lang == "ur" else ans
    return _answer_line(q_num, ans_disp, lang)


ANSWER_RENDERERS = {
    "mcq": render_answer_mcq,
    "fill_blank": render_answer_fill_blank,
    "match_column": render_answer_match_column,
    "short": render_answer_short_detailed,
    "detailed": render_answer_short_detailed,
}


def render_answer_entry(q, q_num, lang="en"):
    renderer = ANSWER_RENDERERS.get(q.get("type"))
    if not renderer:
        return ""
    return renderer(q, q_num, lang)


def build_answer_key_html(questions, worksheet_title, lang="en", as_teacher_cover=True):
    """
    Poori Answer Key ka HTML section banata hai (worksheet ke content ke
    order mein hi numbered — accurate rehta hai).
    """
    title_lbl = L("answer_key", lang)
    title_disp = fix_urdu(worksheet_title) if lang == "ur" else worksheet_title
    title_css = "urdu-text" if lang == "ur" else ""

    items = "".join(render_answer_entry(q, i, lang) for i, q in enumerate(questions, start=1))

    cover = ""
    if as_teacher_cover:
        cover = f"""
        <div style="text-align:center; margin-bottom:20px; border-bottom:3px solid {CONFIG.get('primary_color','#2E86AB')}; padding-bottom:10px;">
            <h2 class="{title_css}">{fix_urdu(L('teacher_copy', lang)) if lang=='ur' else L('teacher_copy', lang)}</h2>
            <p class="{title_css}">{title_disp}</p>
        </div>
        """

    return f"""
    <div class="page answer-key-page">
        {cover}
        <h3 class="{title_css}">{fix_urdu(title_lbl) if lang=='ur' else title_lbl}</h3>
        <div class="answer-key-list">{items}</div>
    </div>
    """


print("✅ Question renderers ready: mcq, fill_blank, match_column, short, detailed.")
print("✅ Answer Key renderer ready (accurate, per question type).")


### Step 8 — Layout Engine (questions/page, spacing, copies, randomization, binding gap, answer key placement)

In [ ]:
# ============================================================
# LAYOUT ENGINE — questions per page, spacing, copies,
# randomization (same/different), binding gap
# ============================================================

def randomize_question_order(questions, mode="same"):
    """
    mode = "same"      -> sab copies mein sawalon ka order same
    mode = "different"  -> har copy mein order shuffle (alag worksheet lage)
    """
    if mode == "different":
        shuffled = questions[:]
        random.shuffle(shuffled)
        return shuffled
    return questions


def paginate(questions, questions_per_page):
    if questions_per_page <= 0:
        return [questions]  # unlimited -> ek hi page
    return [questions[i:i + questions_per_page] for i in range(0, len(questions), questions_per_page)]


def build_worksheet_pages_html(config, questions, worksheet_title, grade, theme,
                                questions_per_page=0, question_spacing_px=18,
                                binding_gap_px=0, lang="en"):
    """
    Ek copy (single worksheet) ke content pages ka HTML banata hai.
    Cover page is function mein include NAHI hota — wo alag call hota hai.
    """
    pages = paginate(questions, questions_per_page)
    total_pages = len(pages)
    html_pages = []

    for idx, page_questions in enumerate(pages, start=1):
        q_html = ""
        counter_start = sum(len(p) for p in pages[:idx - 1]) + 1
        for offset, q in enumerate(page_questions):
            q_num = counter_start + offset
            q_html += render_question(q, q_num, theme)
            q_html += f'<div style="height:{question_spacing_px}px;"></div>'

        page_html = f"""
        <div class="page" style="padding-left:{28 + binding_gap_px}px;">
            {build_page_header(worksheet_title, lang)}
            <div class="questions-area">{q_html}</div>
            {build_page_footer(idx, total_pages, lang)}
        </div>
        """
        html_pages.append(page_html)

    return "".join(html_pages)


def build_full_worksheet_html(config, questions, worksheet_title, grade, subject,
                               theme=None, questions_per_page=0, question_spacing_px=18,
                               num_copies=1, randomize_mode="same", binding_gap_px=0,
                               cover_style=None, lang="en",
                               answer_key_mode="separate"):
    """
    Master function: cover page + N copies + full CSS -> HTML document(s)

    answer_key_mode:
        "inline"   -> Answer Key student worksheet ke HI aakhir mein (same file)
        "separate" -> Answer Key alag teacher-only HTML mein (student file mein NAHI)
        "both"     -> Same file ke aakhir mein bhi, aur alag teacher file bhi

    Returns: dict {"student_html": ..., "teacher_html": ... (ya None)}
    """
    theme = theme or get_theme_for_grade(grade, CONFIG.get("theme_mode", "auto"))
    css = base_css(config["primary_color"], config["accent_color"]) + theme_extra_css(theme, config["accent_color"])

    body_parts = []
    last_copy_questions = None
    for copy_num in range(num_copies):
        copy_questions = randomize_question_order(questions, randomize_mode)
        last_copy_questions = copy_questions
        cover_html = build_cover_page_html(worksheet_title, grade, subject, style=cover_style, lang=lang)
        content_html = build_worksheet_pages_html(
            config, copy_questions, worksheet_title, grade, theme,
            questions_per_page=questions_per_page,
            question_spacing_px=question_spacing_px,
            binding_gap_px=binding_gap_px,
            lang=lang
        )
        body_parts.append(cover_html + content_html)

        # Inline mode: Answer Key har copy ke saath uske turant baad lagegi
        if answer_key_mode in ("inline", "both"):
            body_parts.append(build_answer_key_html(copy_questions, worksheet_title, lang=lang, as_teacher_cover=False))

    student_html = f"""
    <html><head><meta charset="utf-8"><style>{css}</style></head>
    <body>{''.join(body_parts)}</body></html>
    """

    teacher_html = None
    if answer_key_mode in ("separate", "both"):
        answer_section = build_answer_key_html(last_copy_questions, worksheet_title, lang=lang, as_teacher_cover=True)
        teacher_html = f"""
        <html><head><meta charset="utf-8"><style>{css}</style></head>
        <body>{answer_section}</body></html>
        """

    return {"student_html": student_html, "teacher_html": teacher_html}


print("✅ Layout engine ready: pagination, spacing, copies, randomization, binding gap, answer-key modes.")


### Step 9 — Export Engine (PDF + Word)

In [ ]:
# ============================================================
# EXPORT ENGINE — PDF (WeasyPrint) aur Word (python-docx)
# ============================================================

def export_to_pdf(full_html, filename="worksheet.pdf"):
    WeasyHTML(string=full_html).write_pdf(filename)
    print(f"✅ PDF created: {filename}")
    return filename


def export_worksheet_pdf(html_dict, base_filename="worksheet"):
    """
    html_dict = output of build_full_worksheet_html() -> {"student_html":..., "teacher_html":...}
    Returns: {"student_file": "...pdf", "teacher_file": "...pdf" or None}
    """
    student_file = f"{base_filename}.pdf"
    export_to_pdf(html_dict["student_html"], student_file)

    teacher_file = None
    if html_dict.get("teacher_html"):
        teacher_file = f"{base_filename}_ANSWER_KEY.pdf"
        export_to_pdf(html_dict["teacher_html"], teacher_file)

    return {"student_file": student_file, "teacher_file": teacher_file}


def preview_worksheet(full_html, height=650):
    """Shows a worksheet preview right inside Colab (before generating the file)."""
    display(HTML(f'<iframe srcdoc="{full_html.replace(chr(34), "&quot;")}" width="100%" height="{height}px" style="border:1px solid #ccc;"></iframe>'))


def _add_answer_key_to_doc(doc, questions, lang, as_teacher_cover=True):
    if as_teacher_cover:
        h = doc.add_heading(L("teacher_copy", lang), level=0)
        h.alignment = WD_ALIGN_PARAGRAPH.CENTER
    doc.add_heading(L("answer_key", lang), level=1)
    for i, q in enumerate(questions, start=1):
        qtype = q.get("type")
        if qtype == "mcq":
            ans = q.get("answer", "—")
        elif qtype == "fill_blank":
            ans = q.get("answer", "—")
            if isinstance(ans, list):
                ans = ", ".join(ans)
        elif qtype == "match_column":
            left = q.get("left", [])
            right = q.get("right", [])
            ans = ", ".join(f"{l} → {r}" for l, r in zip(left, right))
        else:  # short / detailed
            ans = q.get("answer", "—")
        p = doc.add_paragraph(f"{i}. {ans}")
        if is_urdu_text(str(ans)):
            p.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    doc.add_paragraph("")


def export_to_docx(config, questions, worksheet_title, grade, subject,
                    questions_per_page=0, num_copies=1, randomize_mode="same",
                    lang="en", answer_key_mode="separate",
                    filename="worksheet.docx"):
    """
    answer_key_mode: "inline" | "separate" | "both"
    Returns: dict {"student_file": "...docx", "teacher_file": "...docx" or None}
    """
    doc = Document()
    last_copy_questions = None

    for copy_num in range(num_copies):
        copy_questions = randomize_question_order(questions, randomize_mode)
        last_copy_questions = copy_questions

        # --- Cover page ---
        if config.get("school_logo_base64"):
            try:
                header_img_data = base64.b64decode(config["school_logo_base64"].split(",")[1])
                img_stream = io.BytesIO(header_img_data)
                doc.add_picture(img_stream, width=Inches(1.3))
            except Exception:
                pass
        title = doc.add_heading(config.get("school_name", ""), level=0)
        title.alignment = WD_ALIGN_PARAGRAPH.CENTER
        sub = doc.add_heading(worksheet_title, level=1)
        sub.alignment = WD_ALIGN_PARAGRAPH.CENTER
        info = doc.add_paragraph(f"{L('grade', lang)}: {grade}   |   {L('subject', lang)}: {subject}")
        info.alignment = WD_ALIGN_PARAGRAPH.CENTER
        doc.add_page_break()

        # --- Student info bar ---
        table = doc.add_table(rows=1, cols=4)
        hdr = table.rows[0].cells
        hdr[0].text = f"{L('name', lang)}: ____________"
        hdr[1].text = f"{L('class', lang)}: ______"
        hdr[2].text = f"{L('roll_no', lang)}: ______"
        hdr[3].text = f"{L('date', lang)}: ______"
        doc.add_paragraph("")

        # --- Questions ---
        for i, q in enumerate(copy_questions, start=1):
            qtext = q.get("text", "")
            p = doc.add_paragraph()
            run = p.add_run(f"{i}. {qtext}")
            run.bold = True
            if is_urdu_text(qtext):
                p.alignment = WD_ALIGN_PARAGRAPH.RIGHT

            qtype = q.get("type")
            if qtype == "mcq":
                for j, opt in enumerate(q.get("options", [])):
                    doc.add_paragraph(f"   {chr(97+j)}) {opt}")
            elif qtype == "fill_blank":
                doc.add_paragraph("   " + "____________  " * q.get("blanks", 1))
            elif qtype == "match_column":
                left = q.get("left", [])
                if "_shuffled_right" not in q:
                    right = q.get("right", [])[:]
                    random.shuffle(right)
                    q["_shuffled_right"] = right
                right = q["_shuffled_right"]
                mt = doc.add_table(rows=len(left), cols=3)
                for r, (l, rgt) in enumerate(zip(left, right)):
                    mt.rows[r].cells[0].text = l
                    mt.rows[r].cells[1].text = "—"
                    mt.rows[r].cells[2].text = rgt
            elif qtype in ("short", "detailed"):
                lines = q.get("lines", 3 if qtype == "short" else 6)
                for _ in range(lines):
                    doc.add_paragraph("_" * 60)
            doc.add_paragraph("")

        # Inline answer key right after this copy
        if answer_key_mode in ("inline", "both"):
            doc.add_page_break()
            _add_answer_key_to_doc(doc, copy_questions, lang, as_teacher_cover=False)

        if copy_num < num_copies - 1:
            doc.add_page_break()

    student_filename = filename
    doc.save(student_filename)
    print(f"✅ Word file (student) created: {student_filename}")

    teacher_filename = None
    if answer_key_mode in ("separate", "both"):
        teacher_doc = Document()
        _add_answer_key_to_doc(teacher_doc, last_copy_questions, lang, as_teacher_cover=True)
        teacher_filename = filename.replace(".docx", "_ANSWER_KEY.docx")
        teacher_doc.save(teacher_filename)
        print(f"✅ Word file (teacher answer key) created: {teacher_filename}")

    return {"student_file": student_filename, "teacher_file": teacher_filename}


print("✅ Export engine ready: PDF + DOCX (with Answer Key support: inline / separate / both).")


### Step 10 — 🧪 Demo / Test Run
Runs the core engine with sample questions so you can confirm everything works before
using the real methods below. Each question already has an `answer` field, so the
Answer Key is generated accurately. Set `LANG = "ur"` to preview Urdu output.

In [ ]:
# ============================================================
# DEMO / TEST RUN — used to test the Phase 1 engine
# (Later methods will auto-fill the questions list the same way)
# ============================================================

# Every question now has an "answer" field -> the Answer Key is built accurately from it.
# If the worksheet should be Urdu-medium, set LANG = "ur" (all labels will appear in pure Urdu).

sample_questions = [
    {"type": "mcq", "text": "Which of these is a fruit?",
     "options": ["Apple", "Car", "Chair", "Pen"], "answer": "Apple"},
    {"type": "fill_blank", "text": "A ______ says Meow.", "blanks": 1, "answer": "cat"},
    {"type": "match_column", "text": "Match the animals with their sounds:",
     "left": ["Cat", "Dog", "Cow", "Lion"], "right": ["Meow", "Bark", "Moo", "Roar"]},
    {"type": "short", "text": "Name three colors you see in the sky.", "lines": 3,
     "answer": "Blue, orange, pink (during sunset/sunrise)"},
    {"type": "detailed", "text": "Describe your favorite animal in 5 sentences.", "lines": 6,
     "answer": "(Model answer: student should describe their favorite animal's name, color, habits, food, and one special feature.)"},
    {"type": "mcq", "text": "درست جواب کا انتخاب کریں: سورج کس رنگ کا ہے؟",
     "options": ["پیلا", "نیلا", "سبز", "کالا"], "answer": "پیلا"},
]

WORKSHEET_TITLE = "Science Worksheet — Chapter 2"
GRADE = 3
SUBJECT = "Science"
LANG = "en"                  # "en" ya "ur" — Urdu hone par sab labels khaalis Urdu mein aayenge
ANSWER_KEY_MODE = "separate" # "inline" | "separate" | "both"

html_result = build_full_worksheet_html(
    CONFIG,
    sample_questions,
    worksheet_title=WORKSHEET_TITLE,
    grade=GRADE,
    subject=SUBJECT,
    questions_per_page=3,       # 0 = unlimited (sab ek page par)
    question_spacing_px=20,
    num_copies=2,               # how many copies are needed
    randomize_mode="different", # "same" ya "different"
    binding_gap_px=20,
    lang=LANG,
    answer_key_mode=ANSWER_KEY_MODE,
)

print("🔍 PREVIEW — Student Worksheet (review before generating):")
preview_worksheet(html_result["student_html"])

if html_result.get("teacher_html"):
    print("\n🔑 PREVIEW — Teacher Answer Key (separate file banegi):")
    preview_worksheet(html_result["teacher_html"], height=400)


### Step 11 — 📄 Create Worksheet (from the demo data — choose PDF/Word and Answer Key placement)

In [ ]:
# ============================================================
# CREATE WORKSHEET — Teacher chooses PDF or Word here and
# generates/downloads the final file
# ============================================================

format_dd = widgets.Dropdown(
    options=[('PDF', 'pdf'), ('Word (.docx)', 'docx')],
    value='pdf', description='Save as:', style={'description_width': '90px'}
)

answer_key_dd = widgets.Dropdown(
    options=[('Same file — last page (Inline)', 'inline'),
             ('Separate Teacher-only file', 'separate'),
             ('Dono (Inline + Separate file)', 'both')],
    value='separate', description='Answer Key:', style={'description_width': '90px'},
    layout=widgets.Layout(width='400px')
)

generate_btn = widgets.Button(description='📄 Create Worksheet', button_style='primary')
gen_output = widgets.Output()


def on_generate_click(b):
    with gen_output:
        clear_output()
        ak_mode = answer_key_dd.value

        if format_dd.value == 'pdf':
            html_result = build_full_worksheet_html(
                CONFIG, sample_questions, worksheet_title=WORKSHEET_TITLE,
                grade=GRADE, subject=SUBJECT, questions_per_page=3,
                question_spacing_px=20, num_copies=2, randomize_mode="different",
                binding_gap_px=20, lang=LANG, answer_key_mode=ak_mode,
            )
            files = export_worksheet_pdf(html_result, base_filename="worksheet")
        else:
            files = export_to_docx(
                CONFIG, sample_questions, WORKSHEET_TITLE, GRADE, SUBJECT,
                num_copies=2, randomize_mode="different", lang=LANG,
                answer_key_mode=ak_mode, filename="worksheet.docx"
            )

        print("✅ Student Worksheet:")
        display(FileLink(files["student_file"]))

        if files.get("teacher_file"):
            print("\n🔑 Teacher Answer Key (separate file):")
            display(FileLink(files["teacher_file"]))

        print("\n👆 Use the links above to download your files.")


generate_btn.on_click(on_generate_click)
display(widgets.VBox([format_dd, answer_key_dd, generate_btn, gen_output]))


---
## 📗 Method 2 — Auto File Upload + AI-Generated Worksheet

Upload any file (PDF/Word/TXT) and AI will generate accurate questions and answers from
its content, which then flow into the core engine above.

**Requirement:** You'll need your own **Anthropic (Claude) API key** (from console.anthropic.com).


### Step 12 — Install Method 2 Packages (run once)

In [ ]:
# Extra packages needed for Method 2 (run only once)
!pip install anthropic pdfplumber -q
print("✅ Method 2 packages installed (anthropic + pdfplumber).")


### Step 13 — 📂 File Upload + Text Extraction
Upload a PDF/Word/TXT file, then click **Extract Text**.

In [ ]:
# ============================================================
# METHOD 2 — Step A: File Upload + Text Extraction
# Teacher uploads any file (PDF, Word, or TXT) — its full text
# is extracted and stored in EXTRACTED_TEXT.
# ============================================================

import pdfplumber

EXTRACTED_TEXT = ""  # the full extracted text is stored here


def extract_text_from_pdf(file_bytes):
    text_parts = []
    with pdfplumber.open(io.BytesIO(file_bytes)) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                text_parts.append(t)
    return "\n".join(text_parts)


def extract_text_from_docx(file_bytes):
    d = Document(io.BytesIO(file_bytes))
    return "\n".join(p.text for p in d.paragraphs if p.text.strip())


def extract_text_from_txt(file_bytes):
    for enc in ("utf-8", "utf-16", "cp1256", "latin-1"):
        try:
            return file_bytes.decode(enc)
        except (UnicodeDecodeError, LookupError):
            continue
    return file_bytes.decode("utf-8", errors="ignore")


def extract_text_from_file(filename, file_bytes):
    fname = filename.lower()
    if fname.endswith(".pdf"):
        return extract_text_from_pdf(file_bytes)
    elif fname.endswith(".docx"):
        return extract_text_from_docx(file_bytes)
    else:
        return extract_text_from_txt(file_bytes)


m2_file_uploader = widgets.FileUpload(
    accept='.pdf,.docx,.txt', multiple=False, description='Upload File'
)
m2_extract_btn = widgets.Button(description='📂 Extract Text', button_style='info')
m2_extract_output = widgets.Output()


def on_m2_extract_click(b):
    global EXTRACTED_TEXT
    with m2_extract_output:
        clear_output()
        if not m2_file_uploader.value:
            print("⚠️  Please upload a file first.")
            return
        uploaded = list(m2_file_uploader.value.values())[0] if isinstance(m2_file_uploader.value, dict) else m2_file_uploader.value[0]
        fname = uploaded['name'] if isinstance(uploaded, dict) else uploaded.name
        fbytes = bytes(uploaded['content'] if isinstance(uploaded, dict) else uploaded.content)
        EXTRACTED_TEXT = extract_text_from_file(fname, fbytes)
        print(f"✅ Text extracted: {fname} ({len(EXTRACTED_TEXT)} characters)")
        preview = EXTRACTED_TEXT[:600] + ("..." if len(EXTRACTED_TEXT) > 600 else "")
        print("\n--- Preview ---\n" + preview)


m2_extract_btn.on_click(on_m2_extract_click)
display(widgets.HTML("<h3>📂 Method 2 — File Upload (PDF / Word / TXT)</h3>"))
display(widgets.VBox([m2_file_uploader, m2_extract_btn, m2_extract_output]))


### Step 14 — ⚙️ Worksheet Options
Set the API key, grade, subject, title, language, how many of each question type you
need, and all layout options.

In [ ]:
# ============================================================
# METHOD 2 — Step B: Options (teacher decides here
# what kind of worksheet is needed — it's auto-generated from the file)
# ============================================================

m2_api_key_box = widgets.Password(
    description='Claude API Key:', placeholder='sk-ant-...',
    style={'description_width': '120px'}, layout=widgets.Layout(width='420px')
)

m2_title_box = widgets.Text(
    value='', placeholder='Worksheet ka title (e.g. Chapter 3 - Fractions)',
    description='Title:', style={'description_width': '120px'}, layout=widgets.Layout(width='420px')
)

m2_grade_dd = widgets.Dropdown(
    options=[(f"Grade {i}", i) for i in range(1, 11)],
    value=3, description='Grade:', style={'description_width': '120px'}
)

m2_subject_box = widgets.Text(
    value='', placeholder='e.g. Science, Math, Urdu, English',
    description='Subject:', style={'description_width': '120px'}
)

m2_lang_dd = widgets.Dropdown(
    options=[('English', 'en'), ('Urdu (خالص اردو)', 'ur')],
    value='en', description='Language:', style={'description_width': '120px'}
)

# --- Question type counts (teacher sets however many are needed, 0 = skip) ---
m2_mcq_count = widgets.IntSlider(value=4, min=0, max=20, description='MCQs:', style={'description_width': '120px'})
m2_fill_count = widgets.IntSlider(value=3, min=0, max=20, description='Fill Blanks:', style={'description_width': '120px'})
m2_match_count = widgets.IntSlider(value=1, min=0, max=5, description='Match Columns:', style={'description_width': '120px'})
m2_short_count = widgets.IntSlider(value=2, min=0, max=15, description='Short Q:', style={'description_width': '120px'})
m2_detailed_count = widgets.IntSlider(value=1, min=0, max=10, description='Detailed Q:', style={'description_width': '120px'})

# --- Layout options (compatible with the Phase 1 engine) ---
m2_qpp = widgets.IntText(value=0, description='Q/Page (0=unlimited):', style={'description_width': '160px'})
m2_spacing = widgets.IntSlider(value=18, min=5, max=80, description='Spacing(px):', style={'description_width': '120px'})
m2_copies = widgets.IntText(value=1, description='Copies:', style={'description_width': '120px'})
m2_randomize_dd = widgets.Dropdown(options=[('Same order', 'same'), ('Different (shuffled)', 'different')],
                                    value='same', description='Randomize:', style={'description_width': '120px'})
m2_binding_gap = widgets.IntSlider(value=0, min=0, max=60, description='Binding Gap(px):', style={'description_width': '140px'})
m2_theme_dd = widgets.Dropdown(options=[('Auto (based on grade)', 'auto'), ('Colorful', 'colorful'), ('Professional', 'professional')],
                                value='auto', description='Theme:', style={'description_width': '120px'})
m2_answer_key_dd = widgets.Dropdown(
    options=[('Same file — last page (Inline)', 'inline'),
             ('Separate Teacher-only file', 'separate'),
             ('Both', 'both')],
    value='separate', description='Answer Key:', style={'description_width': '120px'}
)

display(widgets.HTML("<h3>⚙️ Method 2 — Worksheet Options</h3>"))
display(widgets.VBox([
    m2_api_key_box,
    m2_title_box,
    widgets.HBox([m2_grade_dd, m2_subject_box, m2_lang_dd]),
    widgets.HTML("<b>Question Types (how many of each, 0 = skip):</b>"),
    m2_mcq_count, m2_fill_count, m2_match_count, m2_short_count, m2_detailed_count,
    widgets.HTML("<b>Layout:</b>"),
    widgets.HBox([m2_qpp, m2_spacing]),
    widgets.HBox([m2_copies, m2_randomize_dd]),
    widgets.HBox([m2_binding_gap, m2_theme_dd]),
    m2_answer_key_dd,
]))


### Step 15 — 🤖 AI Question Generation
Generates accurate questions and answers strictly from the uploaded file's content.
- Content is based only on the uploaded file — the AI won't invent outside facts
- If Urdu is selected, all output is in pure Urdu script
- Any stray preamble/junk text in the AI's response is automatically cleaned

In [ ]:
# ============================================================
# METHOD 2 — Step C: AI Question Generation
# Sends the extracted file text to the AI, which returns questions +
# accurate answers in our exact JSON schema.
# ============================================================

import json
import anthropic

M2_QUESTIONS = []  # yahan AI ke generate kiye hue questions store honge


def build_ai_prompt(text, grade, subject, lang, counts):
    lang_instruction = (
        "Write entirely in PURE URDU SCRIPT — no word should be in Roman Urdu or "
        "romanized transliteration (not in questions, not in "
        "options mein, na answers mein). Sirf proper nouns (jaise scientific terms "
        "that have no Urdu equivalent) may remain in English."
        if lang == "ur" else
        "Write everything in clear, simple English suitable for the grade level."
    )

    counts_desc = "\n".join(f"- {k}: {v}" for k, v in counts.items() if v > 0)

    return f"""You are an expert curriculum-aligned worksheet writer for school textbooks
(similar in quality to Cambridge, Oxford, or Roots school materials).

SOURCE CONTENT (extracted from the teacher's uploaded file — base ALL questions strictly on this):
---
{text[:12000]}
---

TASK: Generate a worksheet for Grade {grade}, Subject: {subject}.
{lang_instruction}

Generate EXACTLY these question counts (skip a type entirely if its count is 0):
{counts_desc}

Rules:
- Every question must be answerable strictly from the source content above — do not invent facts not present or implied in the text.
- Difficulty and vocabulary must match Grade {grade}.
- Each question must be clearly different from the others (no repeats, no near-duplicates).
- Every question needs an ACCURATE "answer" field — double-check correctness before writing it.
- For match_column, "left" and "right" arrays must be the SAME LENGTH and in correct matching order (left[i] matches right[i]).
- For mcq, provide exactly 4 options, and "answer" must be the exact text of the correct option.

Return ONLY a raw JSON array (no markdown fences, no preamble, no explanation, no trailing text) where each item follows EXACTLY one of these shapes:

{{"type": "mcq", "text": "...", "options": ["...", "...", "...", "..."], "answer": "..."}}
{{"type": "fill_blank", "text": "... ______ ...", "blanks": 1, "answer": "..."}}
{{"type": "match_column", "text": "...", "left": ["...", "..."], "right": ["...", "..."]}}
{{"type": "short", "text": "...", "lines": 3, "answer": "..."}}
{{"type": "detailed", "text": "...", "lines": 6, "answer": "..."}}

Output ONLY the JSON array, nothing else."""


def clean_ai_json_response(raw_text):
    """
    AI kabhi kabhi extra lines add kar deta hai (jaise 'Here are your questions:'
    ya code fences). Ye function un sab ko hata kar sirf saaf JSON nikalta hai.
    """
    t = raw_text.strip()
    # Remove markdown code fences if present
    t = re.sub(r'^```(?:json)?\s*', '', t)
    t = re.sub(r'\s*```$', '', t)
    # Find the first '[' and the last ']' to isolate the JSON array
    start = t.find('[')
    end = t.rfind(']')
    if start != -1 and end != -1 and end > start:
        t = t[start:end + 1]
    return t.strip()


def call_claude_for_questions(api_key, text, grade, subject, lang, counts, model="claude-sonnet-5"):
    client = anthropic.Anthropic(api_key=api_key)
    prompt = build_ai_prompt(text, grade, subject, lang, counts)
    response = client.messages.create(
        model=model,
        max_tokens=4000,
        messages=[{"role": "user", "content": prompt}]
    )
    raw = "".join(block.text for block in response.content if block.type == "text")
    cleaned = clean_ai_json_response(raw)
    questions = json.loads(cleaned)
    return questions


def validate_questions(questions):
    valid = []
    for q in questions:
        t = q.get("type")
        if t == "mcq" and q.get("options") and q.get("answer"):
            valid.append(q)
        elif t == "fill_blank" and q.get("answer"):
            q.setdefault("blanks", 1)
            valid.append(q)
        elif t == "match_column" and q.get("left") and q.get("right") and len(q["left"]) == len(q["right"]):
            valid.append(q)
        elif t in ("short", "detailed") and q.get("answer"):
            q.setdefault("lines", 3 if t == "short" else 6)
            valid.append(q)
    return valid


m2_generate_ai_btn = widgets.Button(description='🤖 Generate Questions with AI', button_style='warning')
m2_ai_output = widgets.Output()


def on_m2_generate_ai_click(b):
    global M2_QUESTIONS
    with m2_ai_output:
        clear_output()
        if not EXTRACTED_TEXT:
            print("⚠️  Please upload a file and extract text first (Step A).")
            return
        if not m2_api_key_box.value:
            print("⚠️  Please enter your Claude API Key.")
            return

        counts = {
            "mcq": m2_mcq_count.value, "fill_blank": m2_fill_count.value,
            "match_column": m2_match_count.value, "short": m2_short_count.value,
            "detailed": m2_detailed_count.value,
        }
        print("⏳ Generating questions with AI, please wait...")
        try:
            raw_questions = call_claude_for_questions(
                m2_api_key_box.value, EXTRACTED_TEXT, m2_grade_dd.value,
                m2_subject_box.value or "General", m2_lang_dd.value, counts
            )
            M2_QUESTIONS = validate_questions(raw_questions)
            print(f"✅ {len(M2_QUESTIONS)} questions generated (validated).")
            for i, q in enumerate(M2_QUESTIONS, 1):
                print(f"  {i}. [{q['type']}] {q['text'][:70]}")
        except json.JSONDecodeError as e:
            print("❌ The AI's response wasn't clean JSON. Please try again. Error:", e)
        except Exception as e:
            print("❌ Error:", e)


m2_generate_ai_btn.on_click(on_m2_generate_ai_click)
display(widgets.HTML("<h3>🤖 Method 2 — AI Question Generation</h3>"))
display(widgets.VBox([m2_generate_ai_btn, m2_ai_output]))


### Step 16 — 📄 Preview & Create Worksheet (Method 2)

In [ ]:
# ============================================================
# METHOD 2 — Step D: Preview + Create Worksheet
# (Reuses the same Phase 1 engine — build_full_worksheet_html,
#  export_worksheet_pdf, export_to_docx)
# ============================================================

m2_preview_btn = widgets.Button(description='🔍 Preview Worksheet', button_style='info')
m2_format_dd = widgets.Dropdown(options=[('PDF', 'pdf'), ('Word (.docx)', 'docx')], value='pdf', description='Save as:')
m2_generate_btn = widgets.Button(description='📄 Create Worksheet', button_style='primary')
m2_final_output = widgets.Output()


def on_m2_preview_click(b):
    with m2_final_output:
        clear_output()
        if not M2_QUESTIONS:
            print("⚠️  Please generate questions with AI first (Step C).")
            return
        result = build_full_worksheet_html(
            CONFIG, M2_QUESTIONS,
            worksheet_title=m2_title_box.value or "Worksheet",
            grade=m2_grade_dd.value, subject=m2_subject_box.value or "General",
            theme=(None if m2_theme_dd.value == "auto" else m2_theme_dd.value),
            questions_per_page=m2_qpp.value, question_spacing_px=m2_spacing.value,
            num_copies=m2_copies.value, randomize_mode=m2_randomize_dd.value,
            binding_gap_px=m2_binding_gap.value, lang=m2_lang_dd.value,
            answer_key_mode=m2_answer_key_dd.value,
        )
        print("🔍 Student Worksheet Preview:")
        preview_worksheet(result["student_html"])
        if result.get("teacher_html"):
            print("\n🔑 Teacher Answer Key Preview:")
            preview_worksheet(result["teacher_html"], height=400)


def on_m2_generate_click(b):
    with m2_final_output:
        clear_output()
        if not M2_QUESTIONS:
            print("⚠️  Please generate questions with AI first (Step C).")
            return

        if m2_format_dd.value == 'pdf':
            result = build_full_worksheet_html(
                CONFIG, M2_QUESTIONS,
                worksheet_title=m2_title_box.value or "Worksheet",
                grade=m2_grade_dd.value, subject=m2_subject_box.value or "General",
                theme=(None if m2_theme_dd.value == "auto" else m2_theme_dd.value),
                questions_per_page=m2_qpp.value, question_spacing_px=m2_spacing.value,
                num_copies=m2_copies.value, randomize_mode=m2_randomize_dd.value,
                binding_gap_px=m2_binding_gap.value, lang=m2_lang_dd.value,
                answer_key_mode=m2_answer_key_dd.value,
            )
            files = export_worksheet_pdf(result, base_filename="method2_worksheet")
        else:
            files = export_to_docx(
                CONFIG, M2_QUESTIONS, m2_title_box.value or "Worksheet",
                m2_grade_dd.value, m2_subject_box.value or "General",
                num_copies=m2_copies.value, randomize_mode=m2_randomize_dd.value,
                lang=m2_lang_dd.value, answer_key_mode=m2_answer_key_dd.value,
                filename="method2_worksheet.docx"
            )

        print("✅ Student Worksheet:")
        display(FileLink(files["student_file"]))
        if files.get("teacher_file"):
            print("\n🔑 Teacher Answer Key:")
            display(FileLink(files["teacher_file"]))


m2_preview_btn.on_click(on_m2_preview_click)
m2_generate_btn.on_click(on_m2_generate_click)
display(widgets.HTML("<h3>📄 Method 2 — Preview & Create</h3>"))
display(widgets.VBox([m2_preview_btn, m2_format_dd, m2_generate_btn, m2_final_output]))


---
## 📘 Method 1 — File + Chapter/Topic/Page, OR Grade/Subject/Topic (No File)

This method offers **two ways** to generate a worksheet — use whichever fits, and simply
leave the other one blank:

- **Option A**: Upload a file and specify a chapter/topic name and/or a page range —
  the system extracts only that portion and sends it to AI
- **Option B**: No file at all — just pick Grade + Subject + Topic, and AI generates the
  worksheet from standard curriculum knowledge for that combination

**Requirement:** Your Claude API key is needed here as well.


### Step 17 — 📂 Option A: File Upload + Chapter/Topic/Page Selection

In [ ]:
# ============================================================
# METHOD 1 — Step A: File Upload + Chapter/Topic/Page Scoping
# Teacher uploads a file, then specifies which chapter/topic
# or page number the worksheet should come from. The system extracts
# only that scoped content and sends it to the AI (not the whole file).
# ============================================================

def extract_pdf_pages(file_bytes, start_page=None, end_page=None):
    """start_page/end_page are 1-indexed, inclusive. None = no limit on that side."""
    text_parts = []
    with pdfplumber.open(io.BytesIO(file_bytes)) as pdf:
        total = len(pdf.pages)
        s = (start_page or 1) - 1
        e = end_page or total
        s = max(0, s)
        e = min(total, e)
        for page in pdf.pages[s:e]:
            t = page.extract_text()
            if t:
                text_parts.append(t)
    return "\n".join(text_parts)


def scope_text_by_topic(full_text, topic_keyword, window_chars=5000):
    """
    Agar teacher ne chapter/topic ka naam diya ho, to us keyword ke aas-paas
    extracts just that portion (not the whole document sent to AI).
    """
    if not topic_keyword or not topic_keyword.strip():
        return full_text
    idx = full_text.lower().find(topic_keyword.strip().lower())
    if idx == -1:
        # keyword not found -> return the full text, the AI can find it itself
        return full_text
    start = max(0, idx - 500)
    end = min(len(full_text), idx + window_chars)
    return full_text[start:end]


M1_EXTRACTED_TEXT = ""  # is method ka scoped/extracted text

m1_file_uploader = widgets.FileUpload(accept='.pdf,.docx,.txt', multiple=False, description='Upload File')
m1_chapter_topic_box = widgets.Text(
    value='', placeholder='e.g. Chapter 3 - Photosynthesis (optional)',
    description='Chapter/Topic:', style={'description_width': '120px'}, layout=widgets.Layout(width='420px')
)
m1_page_start = widgets.IntText(value=0, description='Page From (PDF only, 0=start):', style={'description_width': '200px'})
m1_page_end = widgets.IntText(value=0, description='Page To (PDF only, 0=end):', style={'description_width': '200px'})
m1_extract_btn = widgets.Button(description='📂 Extract Selected Content', button_style='info')
m1_extract_output = widgets.Output()


def on_m1_extract_click(b):
    global M1_EXTRACTED_TEXT
    with m1_extract_output:
        clear_output()
        if not m1_file_uploader.value:
            print("⚠️  Please upload a file first.")
            return
        uploaded = list(m1_file_uploader.value.values())[0] if isinstance(m1_file_uploader.value, dict) else m1_file_uploader.value[0]
        fname = uploaded['name'] if isinstance(uploaded, dict) else uploaded.name
        fbytes = bytes(uploaded['content'] if isinstance(uploaded, dict) else uploaded.content)

        if fname.lower().endswith(".pdf") and (m1_page_start.value > 0 or m1_page_end.value > 0):
            sp = m1_page_start.value if m1_page_start.value > 0 else None
            ep = m1_page_end.value if m1_page_end.value > 0 else None
            raw_text = extract_pdf_pages(fbytes, sp, ep)
            print(f"📄 Extracted pages {sp or 'start'} to {ep or 'end'}.")
        else:
            raw_text = extract_text_from_file(fname, fbytes)

        M1_EXTRACTED_TEXT = scope_text_by_topic(raw_text, m1_chapter_topic_box.value)
        print(f"✅ Content extracted ({len(M1_EXTRACTED_TEXT)} characters).")
        preview = M1_EXTRACTED_TEXT[:600] + ("..." if len(M1_EXTRACTED_TEXT) > 600 else "")
        print("\n--- Preview ---\n" + preview)


m1_extract_btn.on_click(on_m1_extract_click)
display(widgets.HTML("<h3>📂 Method 1 (Option A) — File Upload + Chapter/Topic/Page Selection</h3>"))
display(widgets.VBox([
    m1_file_uploader,
    m1_chapter_topic_box,
    widgets.HBox([m1_page_start, m1_page_end]),
    m1_extract_btn, m1_extract_output
]))


### Step 18 — 🎯 Option B: Grade / Subject / Topic (no file needed)

In [ ]:
# ============================================================
# METHOD 1 — Step B: Grade/Subject/Topic Selection (No File Needed)
# Teacher only selects Grade + Subject + Topic, and AI uses its own
# curriculum knowledge se seedha worksheet bana dega.
# (Option A is the file-based approach, Option B works without a file —
#  leave the one you don't need empty / unused)
# ============================================================

def build_ai_prompt_no_source(grade, subject, topic, lang, counts):
    lang_instruction = (
        "Write entirely in PURE URDU SCRIPT — no word should be in Roman Urdu."
        if lang == "ur" else
        "Write everything in clear, simple English suitable for the grade level."
    )
    counts_desc = "\n".join(f"- {k}: {v}" for k, v in counts.items() if v > 0)

    return f"""You are an expert curriculum-aligned worksheet writer (Cambridge/Oxford/Roots-style
school materials quality).

TASK: Generate a worksheet for Grade {grade}, Subject: {subject}, Topic: {topic}.
Base the questions on standard, widely-taught curriculum content for this grade/subject/topic
combination (the kind found in typical school textbooks for this grade level).
{lang_instruction}

Generate EXACTLY these question counts (skip a type entirely if its count is 0):
{counts_desc}

Rules:
- Difficulty and vocabulary must match Grade {grade}.
- Each question must be clearly different from the others (no repeats, no near-duplicates).
- Every question needs an ACCURATE "answer" field — double-check correctness before writing it.
- For match_column, "left" and "right" arrays must be the SAME LENGTH and in correct matching order.
- For mcq, provide exactly 4 options, and "answer" must be the exact text of the correct option.

Return ONLY a raw JSON array (no markdown fences, no preamble, no explanation) where each item
follows EXACTLY one of these shapes:

{{"type": "mcq", "text": "...", "options": ["...", "...", "...", "..."], "answer": "..."}}
{{"type": "fill_blank", "text": "... ______ ...", "blanks": 1, "answer": "..."}}
{{"type": "match_column", "text": "...", "left": ["...", "..."], "right": ["...", "..."]}}
{{"type": "short", "text": "...", "lines": 3, "answer": "..."}}
{{"type": "detailed", "text": "...", "lines": 6, "answer": "..."}}

Output ONLY the JSON array, nothing else."""


def call_claude_no_source(api_key, grade, subject, topic, lang, counts, model="claude-sonnet-5"):
    client = anthropic.Anthropic(api_key=api_key)
    prompt = build_ai_prompt_no_source(grade, subject, topic, lang, counts)
    response = client.messages.create(model=model, max_tokens=4000, messages=[{"role": "user", "content": prompt}])
    raw = "".join(block.text for block in response.content if block.type == "text")
    cleaned = clean_ai_json_response(raw)
    return json.loads(cleaned)


m1b_grade_dd = widgets.Dropdown(options=[(f"Grade {i}", i) for i in range(1, 11)], value=3,
                                 description='Grade:', style={'description_width': '120px'})
m1b_subject_box = widgets.Text(value='', placeholder='e.g. Science', description='Subject:',
                                style={'description_width': '120px'})
m1b_topic_box = widgets.Text(value='', placeholder='e.g. Photosynthesis', description='Topic:',
                              style={'description_width': '120px'})

display(widgets.HTML("<h3>🎯 Method 1 (Option B) — Grade / Subject / Topic (no file needed)</h3>"))
display(widgets.VBox([widgets.HBox([m1b_grade_dd, m1b_subject_box, m1b_topic_box])]))


### Step 19 — ⚙️ Shared Options (choose your source, question counts, layout)

In [ ]:
# ============================================================
# METHOD 1 — Step C: Shared Options + Source Choice
# Teacher decides here: use Option A (file) or
# Option B (grade/subject/topic dropdown), phir question types
# then sets question counts + layout and generates the worksheet.
# ============================================================

m1_api_key_box = widgets.Password(description='Claude API Key:', placeholder='sk-ant-...',
                                   style={'description_width': '120px'}, layout=widgets.Layout(width='420px'))

m1_source_dd = widgets.Dropdown(
    options=[('Option A — Uploaded File (Chapter/Topic/Page)', 'file'),
             ('Option B — Grade/Subject/Topic (bina file)', 'no_file')],
    value='file', description='Source:', style={'description_width': '120px'}, layout=widgets.Layout(width='420px')
)

m1_title_box = widgets.Text(value='', placeholder='Worksheet title', description='Title:',
                             style={'description_width': '120px'}, layout=widgets.Layout(width='420px'))
m1_lang_dd = widgets.Dropdown(options=[('English', 'en'), ('Urdu (خالص اردو)', 'ur')],
                               value='en', description='Language:', style={'description_width': '120px'})

m1_mcq_count = widgets.IntSlider(value=4, min=0, max=20, description='MCQs:', style={'description_width': '120px'})
m1_fill_count = widgets.IntSlider(value=3, min=0, max=20, description='Fill Blanks:', style={'description_width': '120px'})
m1_match_count = widgets.IntSlider(value=1, min=0, max=5, description='Match Columns:', style={'description_width': '120px'})
m1_short_count = widgets.IntSlider(value=2, min=0, max=15, description='Short Q:', style={'description_width': '120px'})
m1_detailed_count = widgets.IntSlider(value=1, min=0, max=10, description='Detailed Q:', style={'description_width': '120px'})

m1_qpp = widgets.IntText(value=0, description='Q/Page (0=unlimited):', style={'description_width': '160px'})
m1_spacing = widgets.IntSlider(value=18, min=5, max=80, description='Spacing(px):', style={'description_width': '120px'})
m1_copies = widgets.IntText(value=1, description='Copies:', style={'description_width': '120px'})
m1_randomize_dd = widgets.Dropdown(options=[('Same order', 'same'), ('Different (shuffled)', 'different')],
                                    value='same', description='Randomize:', style={'description_width': '120px'})
m1_binding_gap = widgets.IntSlider(value=0, min=0, max=60, description='Binding Gap(px):', style={'description_width': '140px'})
m1_theme_dd = widgets.Dropdown(options=[('Auto (based on grade)', 'auto'), ('Colorful', 'colorful'), ('Professional', 'professional')],
                                value='auto', description='Theme:', style={'description_width': '120px'})
m1_answer_key_dd = widgets.Dropdown(
    options=[('Same file — last page (Inline)', 'inline'),
             ('Separate Teacher-only file', 'separate'),
             ('Both', 'both')],
    value='separate', description='Answer Key:', style={'description_width': '120px'}
)

display(widgets.HTML("<h3>⚙️ Method 1 — Shared Options</h3>"))
display(widgets.VBox([
    m1_api_key_box, m1_source_dd, m1_title_box, m1_lang_dd,
    widgets.HTML("<b>Question Types:</b>"),
    m1_mcq_count, m1_fill_count, m1_match_count, m1_short_count, m1_detailed_count,
    widgets.HTML("<b>Layout:</b>"),
    widgets.HBox([m1_qpp, m1_spacing]),
    widgets.HBox([m1_copies, m1_randomize_dd]),
    widgets.HBox([m1_binding_gap, m1_theme_dd]),
    m1_answer_key_dd,
]))


### Step 20 — 📄 Generate, Preview & Create Worksheet (Method 1)

In [ ]:
# ============================================================
# METHOD 1 — Step D: Generate → Preview → Create Worksheet
# ============================================================

M1_QUESTIONS = []

m1_generate_ai_btn = widgets.Button(description='🤖 Generate Questions', button_style='warning')
m1_preview_btn = widgets.Button(description='🔍 Preview Worksheet', button_style='info')
m1_format_dd = widgets.Dropdown(options=[('PDF', 'pdf'), ('Word (.docx)', 'docx')], value='pdf', description='Save as:')
m1_create_btn = widgets.Button(description='📄 Create Worksheet', button_style='primary')
m1_final_output = widgets.Output()


def _m1_counts():
    return {
        "mcq": m1_mcq_count.value, "fill_blank": m1_fill_count.value,
        "match_column": m1_match_count.value, "short": m1_short_count.value,
        "detailed": m1_detailed_count.value,
    }


def on_m1_generate_ai_click(b):
    global M1_QUESTIONS
    with m1_final_output:
        clear_output()
        if not m1_api_key_box.value:
            print("⚠️  Please enter your Claude API Key.")
            return
        counts = _m1_counts()
        print("⏳ Generating questions with AI...")
        try:
            if m1_source_dd.value == 'file':
                if not M1_EXTRACTED_TEXT:
                    print("⚠️  Please upload and extract a file in Option A first.")
                    return
                raw_questions = call_claude_for_questions(
                    m1_api_key_box.value, M1_EXTRACTED_TEXT, m1b_grade_dd.value,
                    m1b_subject_box.value or "General", m1_lang_dd.value, counts
                )
            else:
                if not (m1b_subject_box.value and m1b_topic_box.value):
                    print("⚠️  Please enter both Subject and Topic for Option B.")
                    return
                raw_questions = call_claude_no_source(
                    m1_api_key_box.value, m1b_grade_dd.value, m1b_subject_box.value,
                    m1b_topic_box.value, m1_lang_dd.value, counts
                )
            M1_QUESTIONS = validate_questions(raw_questions)
            print(f"✅ {len(M1_QUESTIONS)} questions generated (validated).")
            for i, q in enumerate(M1_QUESTIONS, 1):
                print(f"  {i}. [{q['type']}] {q['text'][:70]}")
        except json.JSONDecodeError as e:
            print("❌ The AI's response wasn't clean JSON. Please try again.", e)
        except Exception as e:
            print("❌ Error:", e)


def _m1_build_html():
    return build_full_worksheet_html(
        CONFIG, M1_QUESTIONS,
        worksheet_title=m1_title_box.value or "Worksheet",
        grade=m1b_grade_dd.value, subject=m1b_subject_box.value or "General",
        theme=(None if m1_theme_dd.value == "auto" else m1_theme_dd.value),
        questions_per_page=m1_qpp.value, question_spacing_px=m1_spacing.value,
        num_copies=m1_copies.value, randomize_mode=m1_randomize_dd.value,
        binding_gap_px=m1_binding_gap.value, lang=m1_lang_dd.value,
        answer_key_mode=m1_answer_key_dd.value,
    )


def on_m1_preview_click(b):
    with m1_final_output:
        clear_output()
        if not M1_QUESTIONS:
            print("⚠️  Please generate questions first.")
            return
        result = _m1_build_html()
        print("🔍 Student Worksheet Preview:")
        preview_worksheet(result["student_html"])
        if result.get("teacher_html"):
            print("\n🔑 Teacher Answer Key Preview:")
            preview_worksheet(result["teacher_html"], height=400)


def on_m1_create_click(b):
    with m1_final_output:
        clear_output()
        if not M1_QUESTIONS:
            print("⚠️  Please generate questions first.")
            return
        if m1_format_dd.value == 'pdf':
            result = _m1_build_html()
            files = export_worksheet_pdf(result, base_filename="method1_worksheet")
        else:
            files = export_to_docx(
                CONFIG, M1_QUESTIONS, m1_title_box.value or "Worksheet",
                m1b_grade_dd.value, m1b_subject_box.value or "General",
                num_copies=m1_copies.value, randomize_mode=m1_randomize_dd.value,
                lang=m1_lang_dd.value, answer_key_mode=m1_answer_key_dd.value,
                filename="method1_worksheet.docx"
            )
        print("✅ Student Worksheet:")
        display(FileLink(files["student_file"]))
        if files.get("teacher_file"):
            print("\n🔑 Teacher Answer Key:")
            display(FileLink(files["teacher_file"]))


m1_generate_ai_btn.on_click(on_m1_generate_ai_click)
m1_preview_btn.on_click(on_m1_preview_click)
m1_create_btn.on_click(on_m1_create_click)
display(widgets.HTML("<h3>📄 Method 1 — Generate, Preview & Create</h3>"))
display(widgets.VBox([m1_generate_ai_btn, m1_preview_btn, m1_format_dd, m1_create_btn, m1_final_output]))


---
## 📙 Method 3 — Paste Text (e.g. from ChatGPT) + Auto-Clean — 100% FREE

Paste any text — for example, questions you already generated for free using ChatGPT
(or wrote yourself). The system automatically removes filler lines like *"Here are your
questions:"* or *"Hope this helps!"*, then a **free, rule-based parser** (no API key, no
cost) detects question types (MCQ, Fill in the Blank, Match the Column, Short/Detailed)
and builds the worksheet.

**Best input format:**
```
1. What is photosynthesis?
a) Option one
b) Option two
c) Option three
d) Option four

2. A ______ says Meow.

3. Explain how plants make food.

Answers:
1. a
2. cat
```
Numbering each question, and optionally adding an `Answers:` section at the end, gives
the most accurate results. If an answer can't be found in your pasted text, it will be
clearly flagged (e.g. "⚠️ NOT PROVIDED") instead of guessed — you can fill it in by hand.


### Step 21 — 📝 Paste & Auto-Clean Text

In [ ]:
# ============================================================
# METHOD 3 — Step A: Text Paste + Auto-Clean
# Teacher pastes text copied from anywhere (e.g. ChatGPT).
# System automatically extra/junk lines (jaise "Here are your
# questions:", "Hope this helps!", markdown fences waghera) hata
# and keeps only the actual useful content.
# ============================================================

JUNK_LINE_PATTERNS = [
    r'^\s*(sure|here (are|is)|hope this helps|let me know|i hope|feel free|"?certainly"?)\b.*$',
    r'^\s*```.*$',
    r'^\s*(these|the above|as (you )?(requested|asked))\b.*questions.*$',
    r'^\s*\[?(note|disclaimer)\]?\s*:.*$',
]


def auto_clean_pasted_text(raw_text):
    """
    Har line ko check karta hai — agar wo AI ka 'chatter' (preamble/postamble)
    lagta hai to hata deta hai, sirf asal content rakhta hai.
    """
    lines = raw_text.split("\n")
    cleaned_lines = []
    for line in lines:
        stripped = line.strip()
        if not stripped:
            cleaned_lines.append(line)
            continue
        is_junk = any(re.search(p, stripped, re.IGNORECASE) for p in JUNK_LINE_PATTERNS)
        if not is_junk:
            cleaned_lines.append(line)
    cleaned = "\n".join(cleaned_lines).strip()
    # Extra blank lines simplify karo
    cleaned = re.sub(r'\n{3,}', '\n\n', cleaned)
    return cleaned


M3_RAW_TEXT = ""
M3_CLEANED_TEXT = ""

m3_paste_box = widgets.Textarea(
    value='', placeholder='Paste your text here (e.g. copied from ChatGPT)...',
    layout=widgets.Layout(width='100%', height='200px')
)
m3_clean_btn = widgets.Button(description='🧹 Auto-Clean Text', button_style='info')
m3_clean_output = widgets.Output()


def on_m3_clean_click(b):
    global M3_RAW_TEXT, M3_CLEANED_TEXT
    with m3_clean_output:
        clear_output()
        M3_RAW_TEXT = m3_paste_box.value
        if not M3_RAW_TEXT.strip():
            print("⚠️  Please paste some text first.")
            return
        M3_CLEANED_TEXT = auto_clean_pasted_text(M3_RAW_TEXT)
        removed = len(M3_RAW_TEXT.split("\n")) - len(M3_CLEANED_TEXT.split("\n"))
        print(f"✅ Text cleaned. ({removed} junk lines removed)")
        print("\n--- Cleaned Preview ---\n" + M3_CLEANED_TEXT[:600] + ("..." if len(M3_CLEANED_TEXT) > 600 else ""))


m3_clean_btn.on_click(on_m3_clean_click)
display(widgets.HTML("<h3>📝 Method 3 — Paste Text (e.g. from ChatGPT) + Auto-Clean</h3>"))
display(widgets.VBox([m3_paste_box, m3_clean_btn, m3_clean_output]))


### Step 22 — 🧩 Free Question Parser (rule-based, no API)

In [ ]:
# ============================================================
# METHOD 3 — Step B: FREE Rule-Based Question Parser
# No API key needed. This reads the cleaned pasted text (e.g.
# copied from the free ChatGPT web app) and automatically
# detects question types using pattern matching.
#
# Supported input shapes:
#   1. What is photosynthesis?
#      a) ...   b) ...   c) ...   d) ...
#   2. A ______ says Meow.
#   3. Match the following:
#      Column A: Cat, Dog, Cow
#      Column B: Meow, Bark, Moo
#   4. Explain how plants make food. (short/detailed question)
#
#   Optional answer key section at the end, e.g.:
#   Answers:
#   1. b
#   2. cat
# ============================================================

OPTION_LINE_RE = re.compile(r'^\s*[\(\[]?([a-dA-D])[\)\.\]]\s*(.+)$')
QNUM_RE = re.compile(r'^\s*(\d{1,3})[\.\)]\s*(.*)$')
BLANK_RE = re.compile(r'_{3,}')
ANSWER_SECTION_RE = re.compile(r'^\s*(answers?|answer\s*key)\s*:?\s*$', re.IGNORECASE)
ANSWER_LINE_RE = re.compile(r'^\s*(\d{1,3})[\.\)]\s*(.+)$')
COLUMN_A_RE = re.compile(r'^\s*column\s*a\s*:?\s*(.*)$', re.IGNORECASE)
COLUMN_B_RE = re.compile(r'^\s*column\s*b\s*:?\s*(.*)$', re.IGNORECASE)
MATCH_HEADER_RE = re.compile(r'match\s+the\s+following', re.IGNORECASE)


def split_off_answer_key(text):
    """Separates a trailing 'Answers:' section from the main question text, if present."""
    lines = text.split("\n")
    for i, line in enumerate(lines):
        if ANSWER_SECTION_RE.match(line.strip()):
            main_text = "\n".join(lines[:i]).strip()
            answer_lines = lines[i + 1:]
            answer_map = {}
            for aline in answer_lines:
                m = ANSWER_LINE_RE.match(aline)
                if m:
                    answer_map[int(m.group(1))] = m.group(2).strip()
            return main_text, answer_map
    return text, {}


def split_into_blocks(text):
    """Splits the question text into one block per numbered question."""
    lines = text.split("\n")
    blocks = []
    current_num = None
    current_lines = []
    for line in lines:
        m = QNUM_RE.match(line)
        if m:
            if current_lines:
                blocks.append((current_num, "\n".join(current_lines).strip()))
            current_num = int(m.group(1))
            current_lines = [m.group(2)]
        else:
            current_lines.append(line)
    if current_lines:
        blocks.append((current_num, "\n".join(current_lines).strip()))
    return [b for b in blocks if b[1].strip()]


def parse_block(q_num, block_text, answer_map, default_lines_short=3, default_lines_detailed=6):
    lines = [l for l in block_text.split("\n") if l.strip()]
    if not lines:
        return None
    header_line = lines[0].strip()
    body_lines = lines[1:]

    # --- Match the Column ---
    if MATCH_HEADER_RE.search(header_line):
        left, right = [], []
        for l in body_lines:
            ma = COLUMN_A_RE.match(l.strip())
            mb = COLUMN_B_RE.match(l.strip())
            if ma and ma.group(1):
                left = [x.strip() for x in ma.group(1).split(",") if x.strip()]
            elif mb and mb.group(1):
                right = [x.strip() for x in mb.group(1).split(",") if x.strip()]
        if left and right and len(left) == len(right):
            return {"type": "match_column", "text": header_line, "left": left, "right": right}
        # couldn't parse columns confidently -> fall through to short question

    # --- MCQ (2+ option lines like a) b) c) d)) ---
    options = []
    for l in body_lines:
        m = OPTION_LINE_RE.match(l.strip())
        if m:
            options.append(m.group(2).strip())
    if len(options) >= 2:
        answer_raw = answer_map.get(q_num)
        answer_text = None
        if answer_raw:
            # Answer key might say just a letter (a/b/c/d) or the full option text
            letter_match = re.match(r'^\(?([a-dA-D])\)?\.?$', answer_raw.strip())
            if letter_match:
                idx = ord(letter_match.group(1).lower()) - ord('a')
                if 0 <= idx < len(options):
                    answer_text = options[idx]
            else:
                answer_text = answer_raw.strip()
        return {"type": "mcq", "text": header_line, "options": options,
                "answer": answer_text or "⚠️ NOT PROVIDED — please fill in manually"}

    # --- Fill in the Blank ---
    if BLANK_RE.search(header_line):
        blanks = len(BLANK_RE.findall(header_line))
        answer_raw = answer_map.get(q_num)
        return {"type": "fill_blank", "text": header_line, "blanks": blanks,
                "answer": answer_raw or "⚠️ NOT PROVIDED — please fill in manually"}

    # --- Short / Detailed (fallback) ---
    full_text = " ".join([header_line] + body_lines).strip()
    is_detailed = bool(re.search(r'\b(explain|describe|discuss)\b.*\b(detail|paragraph|essay)\b', full_text, re.IGNORECASE)) \
        or len(full_text) > 140
    answer_raw = answer_map.get(q_num)
    return {
        "type": "detailed" if is_detailed else "short",
        "text": full_text,
        "lines": default_lines_detailed if is_detailed else default_lines_short,
        "answer": answer_raw or "⚠️ NOT PROVIDED — please fill in manually",
    }


def parse_questions_from_text(cleaned_text):
    """
    Main entry point: FREE, no-API parser.
    Returns a list of question dicts in our standard schema.
    Any answer the parser couldn't find is clearly flagged (never invented),
    so the teacher can fill it in by hand if needed.
    """
    main_text, answer_map = split_off_answer_key(cleaned_text)
    blocks = split_into_blocks(main_text)
    questions = []
    for i, (num, block_text) in enumerate(blocks, start=1):
        q_num = num if num is not None else i
        q = parse_block(q_num, block_text, answer_map)
        if q:
            questions.append(q)
    return questions


print("✅ Free rule-based Method 3 parser ready (no API key needed).")


### Step 23 — ⚙️ Options

In [ ]:
# ============================================================
# METHOD 3 — Step C: Options (FREE — no API key needed)
# ============================================================

m3_title_box = widgets.Text(value='', placeholder='Worksheet title', description='Title:',
                             style={'description_width': '120px'}, layout=widgets.Layout(width='420px'))
m3_grade_dd = widgets.Dropdown(options=[(f"Grade {i}", i) for i in range(1, 11)], value=3,
                                description='Grade:', style={'description_width': '120px'})
m3_subject_box = widgets.Text(value='', placeholder='e.g. Science', description='Subject:',
                               style={'description_width': '120px'})
m3_lang_dd = widgets.Dropdown(options=[('English', 'en'), ('Urdu (خالص اردو)', 'ur')],
                               value='en', description='Language:', style={'description_width': '120px'})

m3_qpp = widgets.IntText(value=0, description='Q/Page (0=unlimited):', style={'description_width': '160px'})
m3_spacing = widgets.IntSlider(value=18, min=5, max=80, description='Spacing(px):', style={'description_width': '120px'})
m3_copies = widgets.IntText(value=1, description='Copies:', style={'description_width': '120px'})
m3_randomize_dd = widgets.Dropdown(options=[('Same order', 'same'), ('Different (shuffled)', 'different')],
                                    value='same', description='Randomize:', style={'description_width': '120px'})
m3_binding_gap = widgets.IntSlider(value=0, min=0, max=60, description='Binding Gap(px):', style={'description_width': '140px'})
m3_theme_dd = widgets.Dropdown(options=[('Auto (based on grade)', 'auto'), ('Colorful', 'colorful'), ('Professional', 'professional')],
                                value='auto', description='Theme:', style={'description_width': '120px'})
m3_answer_key_dd = widgets.Dropdown(
    options=[('Same file — last page (Inline)', 'inline'),
             ('Separate Teacher-only file', 'separate'),
             ('Both', 'both')],
    value='separate', description='Answer Key:', style={'description_width': '120px'}
)

display(widgets.HTML("<h3>⚙️ Method 3 — Options (Free — no API key)</h3>"))
display(widgets.VBox([
    m3_title_box,
    widgets.HBox([m3_grade_dd, m3_subject_box, m3_lang_dd]),
    widgets.HTML("<b>Layout:</b>"),
    widgets.HBox([m3_qpp, m3_spacing]),
    widgets.HBox([m3_copies, m3_randomize_dd]),
    widgets.HBox([m3_binding_gap, m3_theme_dd]),
    m3_answer_key_dd,
]))


### Step 24 — 📄 Parse, Preview & Create Worksheet (Method 3)

In [ ]:
# ============================================================
# METHOD 3 — Step D: Parse (FREE) → Preview → Create Worksheet
# ============================================================

M3_QUESTIONS = []

m3_parse_btn = widgets.Button(description='🧩 Parse Questions (Free)', button_style='warning')
m3_preview_btn = widgets.Button(description='🔍 Preview Worksheet', button_style='info')
m3_format_dd = widgets.Dropdown(options=[('PDF', 'pdf'), ('Word (.docx)', 'docx')], value='pdf', description='Save as:')
m3_create_btn = widgets.Button(description='📄 Create Worksheet', button_style='primary')
m3_final_output = widgets.Output()


def on_m3_parse_click(b):
    global M3_QUESTIONS
    with m3_final_output:
        clear_output()
        if not M3_CLEANED_TEXT:
            print("⚠️  Please paste and clean your text first (Step A).")
            return
        M3_QUESTIONS = parse_questions_from_text(M3_CLEANED_TEXT)
        if not M3_QUESTIONS:
            print("⚠️  Couldn't detect any questions. Make sure questions are numbered (1. 2. 3. ...).")
            return
        print(f"✅ {len(M3_QUESTIONS)} questions parsed (free, no API used).")
        for i, q in enumerate(M3_QUESTIONS, 1):
            flag = " ⚠️ answer missing" if "NOT PROVIDED" in str(q.get("answer", "")) else ""
            print(f"  {i}. [{q['type']}] {q['text'][:70]}{flag}")
        if any("NOT PROVIDED" in str(q.get("answer", "")) for q in M3_QUESTIONS):
            print("\nℹ️  Some answers weren't found in your pasted text. They're flagged above —")
            print("   you can add them manually before printing the Answer Key, or paste an")
            print("   'Answers:' section at the end of your text next time.")


def _m3_build_html():
    return build_full_worksheet_html(
        CONFIG, M3_QUESTIONS,
        worksheet_title=m3_title_box.value or "Worksheet",
        grade=m3_grade_dd.value, subject=m3_subject_box.value or "General",
        theme=(None if m3_theme_dd.value == "auto" else m3_theme_dd.value),
        questions_per_page=m3_qpp.value, question_spacing_px=m3_spacing.value,
        num_copies=m3_copies.value, randomize_mode=m3_randomize_dd.value,
        binding_gap_px=m3_binding_gap.value, lang=m3_lang_dd.value,
        answer_key_mode=m3_answer_key_dd.value,
    )


def on_m3_preview_click(b):
    with m3_final_output:
        clear_output()
        if not M3_QUESTIONS:
            print("⚠️  Please parse questions first.")
            return
        result = _m3_build_html()
        print("🔍 Student Worksheet Preview:")
        preview_worksheet(result["student_html"])
        if result.get("teacher_html"):
            print("\n🔑 Teacher Answer Key Preview:")
            preview_worksheet(result["teacher_html"], height=400)


def on_m3_create_click(b):
    with m3_final_output:
        clear_output()
        if not M3_QUESTIONS:
            print("⚠️  Please parse questions first.")
            return
        if m3_format_dd.value == 'pdf':
            result = _m3_build_html()
            files = export_worksheet_pdf(result, base_filename="method3_worksheet")
        else:
            files = export_to_docx(
                CONFIG, M3_QUESTIONS, m3_title_box.value or "Worksheet",
                m3_grade_dd.value, m3_subject_box.value or "General",
                num_copies=m3_copies.value, randomize_mode=m3_randomize_dd.value,
                lang=m3_lang_dd.value, answer_key_mode=m3_answer_key_dd.value,
                filename="method3_worksheet.docx"
            )
        print("✅ Student Worksheet:")
        display(FileLink(files["student_file"]))
        if files.get("teacher_file"):
            print("\n🔑 Teacher Answer Key:")
            display(FileLink(files["teacher_file"]))


m3_parse_btn.on_click(on_m3_parse_click)
m3_preview_btn.on_click(on_m3_preview_click)
m3_create_btn.on_click(on_m3_create_click)
display(widgets.HTML("<h3>📄 Method 3 — Parse, Preview & Create (100% Free)</h3>"))
display(widgets.VBox([m3_parse_btn, m3_preview_btn, m3_format_dd, m3_create_btn, m3_final_output]))


---
## ✅ Summary

All three methods share the same core engine (branding, grade-based themes, question
types, layout controls, Answer Key, PDF/Word export, Urdu support), so a worksheet built
through any method looks and behaves consistently.

| Method | Best for | Cost |
|---|---|---|
| Method 1 | A textbook/file + specific chapter/topic/page — or no file at all, just Grade/Subject/Topic | Needs a Claude API key (a few cents per worksheet; ~$5 free trial credit included) |
| Method 2 | Any file, fully automatic AI-generated worksheet | Needs a Claude API key |
| Method 3 | You already have questions (e.g. from free ChatGPT) and just want them turned into a formatted worksheet | **100% Free** — no API key needed |

Feel free to delete or skip any option/section you don't need in a given run.
